# Optional LAZ Export and Point-Cloud Density Summary

This optional notebook converts ASP stereo point clouds (`*-PC.tif`) to compressed LAZ files with `point2las`, then inspects the exported files with PDAL and writes CSV summaries.

## Requirements

- Ames Stereo Pipeline **3.5.0 or later**
- `point2las` support for `--save-triangulation-error`
- PDAL for the optional metadata and density summary
- Existing ASP point clouds

The notebook reads the central workflow metadata file and automatically retrieves the acquisition name, target CRS, point-cloud directories, and triangulation-error threshold.

### Notes

- `--compressed` creates `.laz` files.
- `--save-triangulation-error` stores triangulation error in the LAS 1.4 `TextureU` field.
- `--max-valid-triangulation-error` takes precedence over percentile-based outlier removal.
- Reported density uses the LAZ bounding-box area and is therefore labelled **bounding-box density**.
- Activate the ASP environment before starting Jupyter, or enable the micromamba wrapper below.


## 1. User settings

Normally, only this cell needs to be edited.

In [ ]:
from pathlib import Path

# ============================================================
# USER SETTINGS
# ============================================================

METADATA_FILE = Path("Metadata-Copy1.sh")

# False: use tools from the active Jupyter environment.
# True : use "micromamba run -n <environment>" for each command.
USE_MICROMAMBA = False
MICROMAMBA_ENV = "asp350_env"

# Search older asp_out/dems outputs as well as the dedicated
# final point-cloud directory.
SEARCH_LEGACY_DEM_OUTPUTS = True
INCLUDE_PRELIMINARY_POINT_CLOUDS = False

# Combine several *-PC.tif tiles from one stereo directory.
GROUP_TILES_BY_STEREO_DIRECTORY = True

# None -> <ASP_OUTPUT_DIR>/laz
LAZ_OUTPUT_DIR_OVERRIDE = None

OVERWRITE_LAZ = False
SAVE_TRIANGULATION_ERROR = True
THREADS = 18
DATUM = "WGS_1984"

# None -> EPSG:<TARGET_EPSG> from the metadata file.
TARGET_SRS = None

# "max_error" | "percentile" | "none"
OUTLIER_MODE = "max_error"

# None -> FINAL_MAX_VALID_TRIANGULATION_ERROR from metadata.
MAX_VALID_TRIANGULATION_ERROR = None
OUTLIER_PERCENTILE = 75.0
OUTLIER_FACTOR = 3.0

EXPORT_MANIFEST_NAME = "laz_export_manifest.csv"
POINT_DENSITY_NAME = "point_density_summary.csv"


## 2. Imports and helper functions

In [ ]:
import json
import re
import shlex
import subprocess
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Optional

import pandas as pd
from IPython.display import display

MINIMUM_ASP_VERSION = (3, 5, 0)


def command_prefix() -> list[str]:
    if USE_MICROMAMBA:
        return ["micromamba", "run", "-n", MICROMAMBA_ENV]
    return []


def tool_command(tool: str, *args: str) -> list[str]:
    return [*command_prefix(), tool, *map(str, args)]


def run_command(command, *, check=True, capture_output=True, cwd=None):
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=capture_output,
        check=False,
    )
    if check and result.returncode != 0:
        cmd = " ".join(shlex.quote(x) for x in command)
        raise RuntimeError(
            f"Command failed ({result.returncode}):\n{cmd}\n\n"
            f"STDOUT:\n{result.stdout}\n\nSTDERR:\n{result.stderr}"
        )
    return result


def parse_version(text: str):
    match = re.search(r"(?<!\\d)(\\d+)\\.(\\d+)\\.(\\d+)(?!\\d)", text)
    return tuple(map(int, match.groups())) if match else None


def read_bash_metadata(metadata_file: Path) -> dict[str, str]:
    """Source metadata in a separate shell and return selected variables."""
    metadata_file = metadata_file.expanduser().resolve()
    if not metadata_file.is_file():
        raise FileNotFoundError(f"Metadata file not found: {metadata_file}")

    keys = [
        "BASE_DIR", "area_name", "TARGET_EPSG", "ASP_OUTPUT_DIR",
        "DSM_OUTPUT_DIR", "FINAL_POINT_CLOUD_OUTPUT_DIR",
        "TRI_MERGED_POINT_CLOUD_OUTPUT_DIR",
        "FINAL_MAX_VALID_TRIANGULATION_ERROR",
    ]
    print_lines = "\\n".join(
        f'printf "__LAZ_META__{key}\\t%s\\n" "${{{key}-}}"' for key in keys
    )
    script = f"source {shlex.quote(str(metadata_file))}\n{print_lines}\n"
    result = subprocess.run(
        ["bash", "-lc", script], text=True, capture_output=True, check=False
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Could not source {metadata_file}\n\n"
            f"STDOUT:\n{result.stdout}\n\nSTDERR:\n{result.stderr}"
        )

    values = {}
    for line in result.stdout.splitlines():
        if line.startswith("__LAZ_META__"):
            key, value = line.removeprefix("__LAZ_META__").split("\t", 1)
            values[key] = value

    missing = [key for key in keys if key not in values]
    if missing:
        raise RuntimeError("Missing metadata variables: " + ", ".join(missing))
    return values


def normalize_product_name(name: str) -> str:
    name = re.sub(r"-PC$", "", name)
    name = re.sub(r"[^A-Za-z0-9._-]+", "_", name)
    return name.strip("_")


def is_preliminary_point_cloud(path: Path) -> bool:
    text = str(path).lower()
    return "preliminary" in text or "stereo_preliminary" in text


def discover_point_clouds(search_roots: Iterable[Path]) -> list[Path]:
    found = {}
    for root in search_roots:
        root = root.expanduser()
        if not root.exists():
            print(f"[INFO] Missing search root: {root}")
            continue
        for path in root.rglob("*-PC.tif"):
            if not path.is_file() or path.stat().st_size == 0:
                continue
            if not INCLUDE_PRELIMINARY_POINT_CLOUDS and is_preliminary_point_cloud(path):
                continue
            found[str(path.resolve())] = path.resolve()
    return sorted(found.values())


@dataclass
class PointCloudProduct:
    product_name: str
    input_files: list[Path]
    output_laz: Path
    log_file: Path


def group_point_clouds(point_clouds, output_dir):
    by_parent = defaultdict(list)
    for path in point_clouds:
        by_parent[path.parent].append(path)

    products = []
    used_names = set()
    for parent, files in sorted(by_parent.items(), key=lambda x: str(x[0])):
        files = sorted(files)
        group_dir = (
            GROUP_TILES_BY_STEREO_DIRECTORY
            and len(files) > 1
            and (parent.name.startswith("stereo_") or parent.name.startswith("merged_"))
        )
        groups = [(parent.name, files)] if group_dir else [
            (normalize_product_name(path.stem), [path]) for path in files
        ]

        for raw_name, group_files in groups:
            name = normalize_product_name(raw_name)
            base = name
            number = 2
            while name in used_names:
                name = f"{base}_{number}"
                number += 1
            used_names.add(name)
            products.append(PointCloudProduct(
                product_name=name,
                input_files=group_files,
                output_laz=output_dir / f"{name}.laz",
                log_file=output_dir / "logs" / f"{name}.point2las.log",
            ))
    return products


def parse_product_fields(product_name: str) -> dict:
    cleaned = re.sub(r"^(stereo|merged)_", "", product_name)
    algorithm_match = re.search(r"_(BM|SGM|MGM)_", cleaned)
    ck_match = re.search(r"_ck(\\d+)", cleaned)
    sk_match = re.search(r"_sk(\\d+)", cleaned)

    configuration = site = algorithm = None
    if algorithm_match:
        algorithm = algorithm_match.group(1)
        prefix = cleaned[:algorithm_match.start()]
        parts = prefix.split("_", 1)
        configuration = parts[0] if parts else None
        site = parts[1] if len(parts) > 1 else None
    else:
        parts = cleaned.split("_", 1)
        configuration = parts[0] if parts else None
        site = parts[1] if len(parts) > 1 else None

    return {
        "configuration": configuration,
        "site": site,
        "algorithm": algorithm,
        "ck": int(ck_match.group(1)) if ck_match else None,
        "sk": int(sk_match.group(1)) if sk_match else None,
    }


def recursive_find_first(obj, candidate_keys):
    if isinstance(obj, dict):
        for key, value in obj.items():
            if str(key).lower() in candidate_keys:
                return value
        for value in obj.values():
            found = recursive_find_first(value, candidate_keys)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = recursive_find_first(value, candidate_keys)
            if found is not None:
                return found
    return None


def find_bounds_object(obj):
    if isinstance(obj, dict):
        lowered = {str(k).lower(): v for k, v in obj.items()}
        if {"minx", "maxx", "miny", "maxy"}.issubset(lowered):
            return lowered
        for value in obj.values():
            found = find_bounds_object(value)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = find_bounds_object(value)
            if found is not None:
                return found
    return None


## 3. Validate ASP, `point2las`, and PDAL

In [ ]:
metadata = read_bash_metadata(METADATA_FILE)

version_result = run_command(tool_command("point2las", "--version"))
version_text = (version_result.stdout + "\n" + version_result.stderr).strip()
asp_version = parse_version(version_text)

if asp_version is None:
    raise RuntimeError(f"Could not parse point2las version:\n{version_text}")
if asp_version < MINIMUM_ASP_VERSION:
    raise RuntimeError(
        "This workflow requires ASP 3.5.0 or later. "
        f"Detected: {'.'.join(map(str, asp_version))}"
    )

help_result = run_command(tool_command("point2las", "--help"), check=False)
help_text = help_result.stdout + "\n" + help_result.stderr
if SAVE_TRIANGULATION_ERROR and "--save-triangulation-error" not in help_text:
    raise RuntimeError("Active point2las does not support --save-triangulation-error")

pdal_result = run_command(tool_command("pdal", "--version"))
pdal_text = (pdal_result.stdout + "\n" + pdal_result.stderr).strip()

print("Environment validation completed")
print("--------------------------------")
print(f"ASP / point2las : {version_text}")
print(f"PDAL            : {pdal_text}")
print(f"Metadata file   : {METADATA_FILE.resolve()}")


## 4. Resolve paths and preview discovered products

In [ ]:
asp_output_dir = Path(metadata["ASP_OUTPUT_DIR"])
final_pc_dir = Path(metadata["FINAL_POINT_CLOUD_OUTPUT_DIR"])
legacy_dem_dir = Path(metadata["DSM_OUTPUT_DIR"])

target_srs = TARGET_SRS or f'EPSG:{metadata["TARGET_EPSG"]}'
max_valid_error = (
    float(metadata["FINAL_MAX_VALID_TRIANGULATION_ERROR"])
    if MAX_VALID_TRIANGULATION_ERROR is None
    else float(MAX_VALID_TRIANGULATION_ERROR)
)

laz_output_dir = (
    asp_output_dir / "laz"
    if LAZ_OUTPUT_DIR_OVERRIDE is None
    else Path(LAZ_OUTPUT_DIR_OVERRIDE).expanduser()
)
laz_output_dir.mkdir(parents=True, exist_ok=True)
(laz_output_dir / "logs").mkdir(parents=True, exist_ok=True)

search_roots = [final_pc_dir]
if SEARCH_LEGACY_DEM_OUTPUTS:
    search_roots.append(legacy_dem_dir)

point_clouds = discover_point_clouds(search_roots)
if not point_clouds:
    raise FileNotFoundError(
        "No non-empty *-PC.tif files found under:\n" +
        "\n".join(f"  - {root}" for root in search_roots)
    )

products = group_point_clouds(point_clouds, laz_output_dir)
manifest_rows = []
for product in products:
    manifest_rows.append({
        "product": product.product_name,
        **parse_product_fields(product.product_name),
        "num_input_files": len(product.input_files),
        "input_files": " | ".join(map(str, product.input_files)),
        "output_laz": str(product.output_laz),
        "already_exists": product.output_laz.exists(),
    })

manifest_df = pd.DataFrame(manifest_rows)
print(f"Area name       : {metadata['area_name']}")
print(f"Target SRS      : {target_srs}")
print(f"Output folder   : {laz_output_dir}")
print(f"Point clouds    : {len(point_clouds)}")
print(f"Export products : {len(products)}")
display(manifest_df)


## 5. Export point clouds to LAZ

Choose one outlier strategy in the settings cell:

- `max_error`: fixed triangulation-error threshold;
- `percentile`: percentile multiplied by a factor;
- `none`: no statistical outlier removal.

Each export receives its own log file under `laz/logs/`.

In [ ]:
def build_point2las_command(product):
    command = tool_command("point2las")
    command.extend(str(path) for path in product.input_files)
    command.extend([
        "--compressed",
        "--datum", DATUM,
        "--t_srs", target_srs,
        "--threads", str(THREADS),
    ])

    if SAVE_TRIANGULATION_ERROR:
        command.append("--save-triangulation-error")

    if OUTLIER_MODE == "max_error":
        if max_valid_error <= 0:
            raise ValueError("Maximum triangulation error must be positive")
        command.extend(["--max-valid-triangulation-error", str(max_valid_error)])
    elif OUTLIER_MODE == "percentile":
        command.extend([
            "--remove-outliers-params",
            str(OUTLIER_PERCENTILE), str(OUTLIER_FACTOR),
        ])
    elif OUTLIER_MODE == "none":
        command.extend(["--remove-outliers-params", "100", "1"])
    else:
        raise ValueError("OUTLIER_MODE must be max_error, percentile, or none")

    command.extend(["-o", str(product.output_laz.with_suffix(""))])
    return command


export_records = []
for index, product in enumerate(products, 1):
    print(f"[{index}/{len(products)}] {product.product_name}", flush=True)
    command_text = ""
    error_message = ""

    if product.output_laz.exists() and not OVERWRITE_LAZ:
        status = "skipped_existing"
        return_code = 0
        print(f"  [SKIP] {product.output_laz}")
    else:
        if product.output_laz.exists():
            product.output_laz.unlink()

        command = build_point2las_command(product)
        command_text = " ".join(shlex.quote(x) for x in command)
        result = run_command(command, check=False)
        return_code = result.returncode

        product.log_file.write_text(
            "COMMAND\n=======\n" + command_text +
            "\n\nSTDOUT\n======\n" + result.stdout +
            "\n\nSTDERR\n======\n" + result.stderr,
            encoding="utf-8",
        )

        if result.returncode == 0 and product.output_laz.is_file():
            status = "created"
            print(f"  [OK] {product.output_laz}")
        else:
            status = "failed"
            error_message = result.stderr.strip() or result.stdout.strip()
            print(f"  [FAILED] See {product.log_file}")

    export_records.append({
        "product": product.product_name,
        **parse_product_fields(product.product_name),
        "num_input_files": len(product.input_files),
        "input_files": " | ".join(map(str, product.input_files)),
        "output_laz": str(product.output_laz),
        "status": status,
        "return_code": return_code,
        "command": command_text,
        "log_file": str(product.log_file),
        "error": error_message,
    })

export_df = pd.DataFrame(export_records)
manifest_csv = laz_output_dir / EXPORT_MANIFEST_NAME
export_df.to_csv(manifest_csv, index=False)
print("\nExport summary")
print(export_df["status"].value_counts(dropna=False))
print(f"Saved manifest: {manifest_csv}")
display(export_df)


## 6. Inspect LAZ files and calculate bounding-box density

In [ ]:
def read_laz_summary(laz_path):
    commands = [
        tool_command("pdal", "info", "--summary", str(laz_path)),
        tool_command("pdal", "info", "--metadata", str(laz_path)),
    ]
    last_error = None
    for command in commands:
        result = run_command(command, check=False)
        if result.returncode != 0:
            last_error = result.stderr.strip() or result.stdout.strip()
            continue
        try:
            payload = json.loads(result.stdout)
        except json.JSONDecodeError as exc:
            last_error = str(exc)
            continue

        count = recursive_find_first(
            payload, {"num_points", "count", "point_count", "numpoints"}
        )
        bounds = find_bounds_object(payload)
        if count is not None and bounds is not None:
            return {
                "num_points": int(count),
                "minx": float(bounds["minx"]),
                "maxx": float(bounds["maxx"]),
                "miny": float(bounds["miny"]),
                "maxy": float(bounds["maxy"]),
            }
    raise RuntimeError(f"Could not read {laz_path}: {last_error}")


density_records = []
available_laz = [
    Path(path)
    for path in export_df.loc[
        export_df["status"].isin(["created", "skipped_existing"]), "output_laz"
    ]
    if Path(path).is_file()
]

for index, laz_path in enumerate(available_laz, 1):
    print(f"[{index}/{len(available_laz)}] {laz_path.name}")
    fields = parse_product_fields(laz_path.stem)
    try:
        summary = read_laz_summary(laz_path)
        width = summary["maxx"] - summary["minx"]
        height = summary["maxy"] - summary["miny"]
        area_m2 = width * height
        density = summary["num_points"] / area_m2 if area_m2 > 0 else float("nan")
        density_records.append({
            "product": laz_path.stem,
            **fields,
            "laz_file": str(laz_path),
            **summary,
            "bbox_width_m": width,
            "bbox_height_m": height,
            "bbox_area_km2": area_m2 / 1_000_000,
            "bbox_density_pts_per_m2": density,
            "status": "ok",
            "error": "",
        })
    except Exception as exc:
        density_records.append({
            "product": laz_path.stem,
            **fields,
            "laz_file": str(laz_path),
            "status": "failed",
            "error": str(exc),
        })

density_df = pd.DataFrame(density_records)
if not density_df.empty:
    sort_cols = [c for c in ["site", "configuration", "algorithm", "ck", "sk"] if c in density_df]
    if sort_cols:
        density_df = density_df.sort_values(sort_cols, na_position="last").reset_index(drop=True)

density_csv = laz_output_dir / POINT_DENSITY_NAME
density_df.to_csv(density_csv, index=False)
print(f"Saved point-density summary: {density_csv}")
display(density_df)


## 7. Output structure

```text
<ASP_OUTPUT_DIR>/
└── laz/
    ├── <product>.laz
    ├── laz_export_manifest.csv
    ├── point_density_summary.csv
    └── logs/
        └── <product>.point2las.log
```

This optional step does not modify the original ASP point-cloud rasters.
